In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
import albumentations as A
import numpy as np
import os
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras import mixed_precision
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.applications import ResNet50V2
from tensorflow.keras.applications.resnet_v2 import preprocess_input
mixed_precision.set_global_policy('mixed_float16')
tf.keras.backend.clear_session()



c:\Users\ASUS\miniconda3\envs\tf\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA GeForce RTX 3050 Laptop GPU, compute capability 8.6


In [2]:
DATA_ROOT = "Processed_AffectNet" 
TRAIN_DIR = os.path.join(DATA_ROOT, "train")
VAL_DIR = os.path.join(DATA_ROOT, "valid")

BATCH_SIZE = 30
IMG_SIZE = 224
NUM_CLASSES = 8 
EPOCHS = 40


In [ ]:
aug_pipeline = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.GaussNoise(p=0.2),
    A.CoarseDropout(max_holes=8, max_height=20, max_width=20, p=0.2),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=15, p=0.5),
])

def aug_fn(image):
    data = {"image": image}
    aug_data = aug_pipeline(**data)
    aug_img = aug_data["image"]
    aug_img = tf.cast(aug_img, tf.float32)
    return aug_img


def process_train_data(image, label):
    aug_img = tf.numpy_function(func=aug_fn, inp=[image], Tout=tf.float32)
    
    aug_img = preprocess_input(aug_img)
    
    aug_img.set_shape((IMG_SIZE, IMG_SIZE, 3))
    return aug_img, label

def process_val_data(image, label):
    image = tf.cast(image, tf.float32)
    image = preprocess_input(image) 
    return image, label

C:\Users\ASUS\AppData\Local\Temp\ipykernel_10812\2761114416.py:5: UserWarning: Argument(s) 'max_holes, max_height, max_width' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=8, max_height=20, max_width=20, p=0.2),
c:\Users\ASUS\miniconda3\envs\tf\lib\site-packages\albumentations\core\validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [4]:
train_ds_batched = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    shuffle=True,
    seed=123,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    shuffle=False,
    seed=123,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

train_ds = train_ds_batched.unbatch()
train_ds = train_ds.map(process_train_data, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.batch(BATCH_SIZE)

val_ds = val_ds.map(process_val_data, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

Found 17063 files belonging to 8 classes.
Found 5394 files belonging to 8 classes.


In [ ]:
def build_model():
    
    base_model = ResNet50V2(include_top=False, weights='imagenet', input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base_model.trainable = False 

    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base_model(inputs, training=False) 
    x = layers.GlobalAveragePooling2D()(x)
    
    x = layers.Dense(256, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x) 
    x = layers.Dropout(0.5)(x)
    
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)

    model = models.Model(inputs, outputs)
    return model, base_model

model, base_model = build_model()

loss_fn = tf.keras.losses.CategoricalCrossentropy()

In [6]:
print("--- Stage 1: Training Top Layers ---")
model.compile(optimizer=optimizers.Adam(learning_rate=1e-3), loss=loss_fn, metrics=['accuracy'])


history_warmup = model.fit(
    train_ds,
    validation_data=val_ds, 
    epochs=30,
)


--- Stage 1: Training Top Layers ---
Epoch 1/30
569/569 [==============================] - 62s 99ms/step - loss: 2.0760 - accuracy: 0.2400 - val_loss: 1.5395 - val_accuracy: 0.4253
Epoch 2/30
569/569 [==============================] - 55s 96ms/step - loss: 1.8544 - accuracy: 0.2822 - val_loss: 1.4797 - val_accuracy: 0.4483
Epoch 3/30
569/569 [==============================] - 55s 96ms/step - loss: 1.7881 - accuracy: 0.3033 - val_loss: 1.4560 - val_accuracy: 0.4638
Epoch 4/30
569/569 [==============================] - 55s 96ms/step - loss: 1.7605 - accuracy: 0.3126 - val_loss: 1.4484 - val_accuracy: 0.4622
Epoch 5/30
569/569 [==============================] - 55s 96ms/step - loss: 1.7349 - accuracy: 0.3271 - val_loss: 1.4326 - val_accuracy: 0.4655
Epoch 6/30
569/569 [==============================] - 55s 96ms/step - loss: 1.7384 - accuracy: 0.3276 - val_loss: 1.4153 - val_accuracy: 0.4718
Epoch 7/30
569/569 [==============================] - 51s 89ms/step - loss: 1.7218 - accuracy: 0.32

In [9]:

class_names = sorted(os.listdir(TRAIN_DIR)) 

y_train = []
print("Calculating Class Weights...")
for i, class_name in enumerate(class_names):
    folder_path = os.path.join(TRAIN_DIR, class_name)
    count = len(os.listdir(folder_path))
    print(f" - {class_name}: {count} images")
    y_train.extend([i] * count)

class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(enumerate(class_weights_array))
print(f"Computed Weights: {class_weights}")

Calculating Class Weights...
 - Anger: 2335 images
 - Contempt: 1991 images
 - Disgust: 2239 images
 - Fear: 2013 images
 - Happy: 2153 images
 - Neutral: 1612 images
 - Sad: 1910 images
 - Surprise: 2810 images
Computed Weights: {0: 0.9134368308351177, 1: 1.071258161727775, 2: 0.9526016078606521, 3: 1.0595504222553402, 4: 0.9906525777984209, 5: 1.3231234491315136, 6: 1.1166884816753926, 7: 0.7590302491103202}


In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-7, verbose=1),
    ModelCheckpoint("ResNet50V2.h5", monitor="val_accuracy", save_best_only=True, verbose=1)
]

base_model.trainable = True

for layer in base_model.layers:
    layer.trainable = True

for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False


model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5), 
    loss=loss_fn, 
    metrics=['accuracy']
)

history_finetune = model.fit(
    train_ds,
    class_weight=class_weights,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks = callbacks
)
model.save("Final_ResNet50V2.h5")

Epoch 1/40
    569/Unknown - 105s 168ms/step - loss: 1.5413 - accuracy: 0.4095
Epoch 1: val_accuracy improved from -inf to 0.57935, saving model to ResNet50V2.h5
569/569 [==============================] - 116s 187ms/step - loss: 1.5413 - accuracy: 0.4095 - val_loss: 1.1271 - val_accuracy: 0.5793 - lr: 1.0000e-05
Epoch 2/40
569/569 [==============================] - ETA: 0s - loss: 1.4474 - accuracy: 0.4420
Epoch 2: val_accuracy improved from 0.57935 to 0.59436, saving model to ResNet50V2.h5
569/569 [==============================] - 105s 185ms/step - loss: 1.4474 - accuracy: 0.4420 - val_loss: 1.0870 - val_accuracy: 0.5944 - lr: 1.0000e-05
Epoch 3/40
569/569 [==============================] - ETA: 0s - loss: 1.3908 - accuracy: 0.4612
Epoch 3: val_accuracy did not improve from 0.59436
569/569 [==============================] - 103s 181ms/step - loss: 1.3908 - accuracy: 0.4612 - val_loss: 1.0645 - val_accuracy: 0.5940 - lr: 1.0000e-05
Epoch 4/40
569/569 [==============================] -